In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import intrasom

# Settings
pd.set_option('display.max_columns', None)

In [ ]:
# ----------- Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()
# Da wir jetzt im Unterordner '3.2_Machine-Learning' sind, ist Preprocessing in '3.1_Preprocessing/Preprocessing'
preprocessing_root = base_dir.parent.parent / "3.1_Preprocessing" / "Preprocessing"

# ----------- Suche nach dem neusten Preprocessing-Ordner
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")

# Parsing des Preprocessing-Zeitstempels aus dem Ordnernamen
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)

# ----------- Lade Preprocessed Data
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_prep = pd.read_csv(input_path_prep, low_memory=False)


# ----------- Daten sind bereits komplett (inkl. Temperatur)
df_full = df_prep
print(f"Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: {df_full.shape}")


<div class="alert alert-info">
    <b>IntraSOM Machine Learning</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Quantile Gaussian) + Temperatur (Cleaned)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Library:</b><br>
    - <a href="https://github.com/InTRA-USP/IntraSOM">IntraSOM</a>
</div>

In [ ]:
# ----------- Features auswählen
target_ions = ["Na", "Ca", "Mg", "Cl", "HCO3", "SO4"]
features = []
for col in df_full.columns:
    if "_log_gauss" in col:
        element = col.split("_")[0]
        if element in target_ions:
            features.append(col)

print(f"Input Features für SOM: {features}")

# ----------- Filtern
df_som = df_full[features + ['temperature_in_c']].copy()

# ---------- WICHTIG: Temperatur in Numeric wandeln (Fehler abfangen)
df_som['temperature_in_c'] = pd.to_numeric(df_som['temperature_in_c'], errors='coerce')

# ---------- FILTER: Nur Temperaturen > 10 Grad
len_before_temp = len(df_som)
df_som = df_som[df_som['temperature_in_c'] > 10]
print(f"Filter Temp > 10°C: {len_before_temp - len(df_som)} Zeilen entfernt. Verbleibend: {len(df_som)}")

initial_len = len(df_som)
df_som.dropna(subset=features, inplace=True)
print(f"Zeilen mit fehlenden Features entfernt: {initial_len - len(df_som)}. Final: {len(df_som)}")

In [ ]:
# ----------- IntraSOM erwartet Input Data
# Da unsere Features bereits _log_gauss (normalverteilt) transformiert sind, 
# müssen wir sie nicht zwingend auf [0,1] MinMax scalen, wie es MiniSom manchmal mag.
# IntraSOM kann 'normalization' Parameter selbst handeln (z.B. 'var' = Varianz).
# Wir übergeben die Daten vorerst wie sie sind (Quasi-Standard-Normalverteilung),
# da sie durch QuantileTransformer schon skaliert wurden.

data_values = df_som[features].values
print(f"Data Shape für IntraSOM: {data_values.shape}")

In [ ]:
# ----------- IntraSOM Training
if 'som' in locals():
    del som # Reset falls cell re-run

# Map Config
mapsize = (10, 10)

# Build SOM
print("Building IntraSOM model...")
som = intrasom.SOMFactory.build(
    data=data_values,
    mapsize=mapsize,
    mapshape='toroid',    # Wie im Github Example
    lattice='hexa',       # Hexagonal gewünscht
    normalization=None,   # Features sind schon preprocessed (Mean~0, Std~1)
    initialization='random',
    neighborhood='gaussian',
    training='batch',
    name='IntraSOM_Analysis',
    component_names=features,
    missing=False         # Wir haben NAs vorher entfernt
)

print("Starting training...")
som.train()

In [ ]:
# ----------- Visualization Output Folder
output_root = base_dir / "IntraSom_Results"
output_root.mkdir(exist_ok=True)

# ----------- Visualization
# IntraSOM bietet eigene Plotting Tools via PlotFactory
print("Training complete. Generating visualizations...")

import intrasom.visualization
plot_factory = intrasom.visualization.PlotFactory(som)

# 1. U-Matrix
print("Plotting U-Matrix...")
plot_factory.plot_umatrix(
    hits=True, 
    title="U-Matrix (Distance Map) - Hexagonal"
)
# Speichern des aktuellen Plots (da IntraSOM das ggf. nur anzeigt)
timestamp = datetime.now().strftime("%H-%M-%S")
plt.savefig(output_root / f"IntraSOM_UMatrix_{timestamp}.png")
plt.show()

print(f"Results should be saved in: {output_root}")

# ----------- Setup für PDF Report (Analog zu MiniSom)

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import matplotlib.pyplot as plt

# ----------- Plot Funktion (aus MiniSom Notebook)
def plot_hex_map_to_axis(ax, data_matrix, title, cmap='viridis', label_suffix='', annotate=False):
    sy, sx = data_matrix.shape
    ax.set_aspect('equal')
    patches = []
    colors = []
    
    for y in range(sy):
        for x in range(sx):
            val = data_matrix[y, x]
            if np.isnan(val): 
                continue
                
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            
            hex_poly = mpatches.RegularPolygon(
                (center_x, center_y), numVertices=6, radius=radius, 
                orientation=np.radians(30), edgecolor='k', linewidth=0.5
            )
            patches.append(hex_poly)
            colors.append(val)
            
            if annotate:
                ax.text(center_x, center_y, f"{val:.1f}", ha='center', va='center', 
                        fontsize=7, color='black', fontweight='bold')
            
    if not patches:
        return

    collection = PatchCollection(patches, cmap=cmap, alpha=0.9)
    collection.set_array(np.array(colors))
    ax.add_collection(collection)
    
    ax.set_xlim(-0.5, sx + 0.5)
    ax.set_ylim(-0.5, sy * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return collection

In [ ]:
# ----------- Ergebnisse aus IntraSom extrahieren

# 1. Results DataFrame abrufen (Enthält Mapping Sample -> BMU)
results_df = som.results_dataframe

# 2. Koordinaten auf df_som mappen
# Hinweis: Zeilen entsprechen der Input-Reihenfolge (df_som)
df_som['som_x'] = results_df['Ret_x'].values.astype(int)
df_som['som_y'] = results_df['Ret_y'].values.astype(int)

# 3. Neurons DataFrame abrufen (Enthält Gewichte und U-Dist)
neurons_df = som.neurons_dataframe

# ----------- Matrizen für Plotting vorbereiten
som_x = 10
som_y = 10

# Helper um DataFrame Spalte zu Matrix zu pivotieren
def get_matrix_from_neurons(col_name):
    # neurons_df has Ret_x, Ret_y. We pivot.
    # Fillna just in case, though usually full.
    matrix = neurons_df.pivot(index='Ret_y', columns='Ret_x', values=col_name).values
    return matrix

# U-Matrix (Als 'Udist' in neurons_df verfügbar, skaliert 0-1)
u_matrix = get_matrix_from_neurons('Udist')

# Mittlere Temperaturkarte
temp_map = np.full((som_y, som_x), np.nan)
agg = df_som.groupby(['som_y', 'som_x'])['temperature_in_c'].mean()
for (gy, gx), val in agg.items():
    if 0 <= gy < som_y and 0 <= gx < som_x:
        temp_map[gy, gx] = val

print("Daten extrahiert und Matrizen vorbereitet.")

In [ ]:
# ----------- PDF Report erstellen
output_root = base_dir / "IntraSom_Results"
output_root.mkdir(exist_ok=True)
pdf_path = output_root / f"IntraSOM_Temperature_Report_{latest_folder.name}.pdf"

print(f"Erstelle PDF Report: {pdf_path} ...")

with PdfPages(pdf_path) as pdf:
    
    # 1. Deckblatt
    plt.figure(figsize=(8, 6))
    plt.text(0.5, 0.5, "IntraSOM Temperatur Analyse Report\n\nHexagonal SOM (10x10)\nLibrary: IntraSOM\nInput: Major Ions (Log/Gauss)\nOverlay: Temperatur (Mean & Dist)\nFILTER: Nur Temp > 10°C", 
             ha='center', va='center', fontsize=20)
    plt.axis('off')
    pdf.savefig()
    plt.close()
    
    # 2. U-Matrix
    fig, ax = plt.subplots(figsize=(8, 8))
    col_u = plot_hex_map_to_axis(ax, u_matrix, "U-Matrix (Distance Map)", cmap='bone_r')
    plt.colorbar(col_u, ax=ax, fraction=0.046, pad=0.04, label='Norm. Distance')
    pdf.savefig(fig)
    plt.close(fig)
    
    # 3. Mean Temperature Map
    fig, ax = plt.subplots(figsize=(10, 10))
    col_t = plot_hex_map_to_axis(ax, temp_map, "Mean Temperature per Cluster (> 10°C)", cmap='coolwarm', annotate=True)
    plt.colorbar(col_t, ax=ax, fraction=0.046, pad=0.04, label='Temp °C')
    pdf.savefig(fig)
    plt.close(fig)
    
    # 4. Component Planes
    # In 'neurons_df', columns are named 'B_Name'
    # We iterate over 'features' list (e.g. 'Na_log_gauss') -> look for 'B_Na_log_gauss'
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    
    # Use the features list defined earlier in the notebook
    for i, feature in enumerate(features):
        col_name = f"B_{feature}"
        if col_name in neurons_df.columns:
            mat = get_matrix_from_neurons(col_name)
            short_name = feature.split("_")[0]
            plot_hex_map_to_axis(axes[i], mat, short_name, cmap='viridis')
        else:
            axes[i].text(0.5, 0.5, "Not Found", ha='center')
            
    plt.suptitle("Komponenten-Ebenen (Gewichte)")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
    
    # 5. Detail Distributions
    # Sort clusters by size
    cluster_counts = df_som.groupby(['som_x', 'som_y']).size().sort_values(ascending=False)
    all_clusters = cluster_counts.index.tolist()
    
    plots_per_page = 9
    chunks = [all_clusters[i:i + plots_per_page] for i in range(0, len(all_clusters), plots_per_page)]
    
    print(f"Erstelle Verteilungs-Seiten: {len(chunks)} Seiten für {len(all_clusters)} Cluster...")
    
    for page_idx, chunk in enumerate(chunks):
        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        axes = axes.flatten()
        plt.suptitle(f"Temp Distribution (>10°C) - Page {page_idx + 1}/{len(chunks)}", fontsize=16)
        
        for i, (cx, cy) in enumerate(chunk):
            # Note: cx=som_x=Ret_x, cy=som_y=Ret_y
            subset = df_som[(df_som['som_x'] == cx) & (df_som['som_y'] == cy)]
            temps = subset['temperature_in_c'].dropna()
            
            ax = axes[i]
            if len(temps) == 0:
                ax.text(0.5, 0.5, "No Data", ha='center')
            else:
                sns.histplot(temps, kde=True, ax=ax, color='crimson', kde_kws={'cut': 0})
            
            ax.set_title(f"Cluster ({cx}, {cy}) - N={len(temps)}")
            ax.set_xlabel("Temp °C")

        # Hide empty axes
        for j in range(len(chunk), plots_per_page):
            axes[j].axis('off')
            
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pdf.savefig(fig)
        plt.close(fig)

print("PDF Report fertiggestellt.")